# Goal #6 — Major-Class Classification (Scopus + Patents, Abstract-only)

This notebook implements **Goal #6**:

- Classify **Scopus publications** and **US patents** into the **5 major classes**:
  - Material
  - Computation
  - Experimentation
  - Application
  - Review/Book
- Output **Primary / Secondary / Tertiary** major-class predictions (plus scores).
- **Critical constraint:** classification is based **exclusively on the Abstract** (not Title / Keywords / CPC / metadata).

Inputs in this repo:
- `raw_data/MANI_KW_PAPERS_scopus.csv`
- `raw_data/MANI_KW_PATENTS_A_weds1969to2009.csv`
- `raw_data/MANI_KW_PATENTS_B_weds2010tonow.csv`
- `classification/MAJOR_CLASSES.md`
- `docs/INSTRUCTIONS.md`
- `docs/AI_CLASSIFICATON_IS421_2026S.md`

Outputs (written under `outputs/goal6/`):
- `goal6_scopus_major_classified.csv`
- `goal6_patents_major_classified.csv`
- metrics and spot-check tables (optional)

In [ ]:
# Mount Drive (Google Colab)
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/mani

## 1) Runtime Setup (paths, installs, reproducibility)

Colab free tier + zero-cost OSS tooling only. This section installs dependencies, sets paths, and defines a shared `CONFIG` dict.

In [ ]:
# Dependency setup for Colab (avoid breaking Colab's pinned scientific stack)
#
# NOTE:
# - Do NOT `pip -U` core packages like numpy/pandas in Colab.
# - If you previously upgraded them and hit import errors, this cell will
#   pin back to compatible versions ONCE and then restart the runtime.

import os
import re
import json
import sys
import subprocess
from pathlib import Path


def pip_install(*args: str) -> None:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    subprocess.check_call(cmd)


SENTINEL = Path('/content/.mani_deps_ok')

if not SENTINEL.exists():
    # Pin to Colab-compatible versions to avoid SciPy / TF / other conflicts.
    # (Colab currently expects pandas==2.2.2 and numpy < 2.2.)
    pip_install('pandas==2.2.2', 'numpy<2.2', 'scipy<1.13')

    # Install Hugging Face runtime deps.
    pip_install('-U', 'transformers', 'accelerate', 'sentencepiece', 'tqdm', 'datasets')

    SENTINEL.write_text('ok', encoding='utf-8')
    print('Installed/pinned dependencies. Restarting runtime...')
    os._exit(0)
else:
    # Normal path on subsequent runs (no restarts).
    pip_install('-U', 'transformers', 'accelerate', 'sentencepiece', 'tqdm', 'datasets')

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import pipeline

# Reproducibility
SEED = 421
np.random.seed(SEED)
torch.manual_seed(SEED)

# Resolve project root robustly (works even if you ran this cell before the %cd cell)
CWD = Path('.').resolve()
DRIVE_ROOT = Path('/content/drive/MyDrive/mani')
ROOT = DRIVE_ROOT if DRIVE_ROOT.exists() else CWD

OUT_DIR = ROOT / 'outputs' / 'goal6'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'scopus_csv': str(ROOT / 'raw_data' / 'MANI_KW_PAPERS_scopus.csv'),
    'patents_csv_a': str(ROOT / 'raw_data' / 'MANI_KW_PATENTS_A_weds1969to2009.csv'),
    'patents_csv_b': str(ROOT / 'raw_data' / 'MANI_KW_PATENTS_B_weds2010tonow.csv'),
    'major_classes_md': str(ROOT / 'classification' / 'MAJOR_CLASSES.md'),
    'out_dir': str(OUT_DIR),
    'out_csv_scopus': str(OUT_DIR / 'goal6_scopus_major_classified.csv'),
    'out_csv_patents': str(OUT_DIR / 'goal6_patents_major_classified.csv'),
    'model_name': 'typeform/distilbert-base-uncased-mnli',
    'batch_size': 16,
    'hypothesis_template': 'This text is about {}.',
}

print('CWD:', CWD)
print('ROOT:', ROOT)
print('DRIVE_ROOT exists:', DRIVE_ROOT.exists())
print('OUT_DIR:', OUT_DIR)
print('Versions:', {'numpy': np.__version__, 'pandas': pd.__version__})
print(json.dumps(CONFIG, indent=2))

## 2) Load & Display Assignment Instructions from `docs/`

These are included in-notebook for traceability and to keep constraints visible while you work.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

# Make this cell resilient if it runs before the runtime-setup cell
if 'ROOT' not in globals():
    DRIVE_ROOT = Path('/content/drive/MyDrive/mani')
    ROOT = DRIVE_ROOT if DRIVE_ROOT.exists() else Path('.').resolve()

instructions_path = ROOT / 'docs' / 'INSTRUCTIONS.md'
assignment_path = ROOT / 'docs' / 'AI_CLASSIFICATON_IS421_2026S.md'

if not instructions_path.exists() or not assignment_path.exists():
    raise FileNotFoundError(
        'Could not find docs files. '
        f'ROOT={ROOT} | CWD={Path(".").resolve()} | '
        f'INSTRUCTIONS exists={instructions_path.exists()} | '
        f'ASSIGNMENT exists={assignment_path.exists()}\n'
        'In Colab: run the mount/%cd cell first, then re-run the runtime setup cell.'
    )

instructions_text = instructions_path.read_text(encoding='utf-8', errors='ignore')
assignment_text = assignment_path.read_text(encoding='utf-8', errors='ignore')

display(Markdown('## docs/INSTRUCTIONS.md'))
display(Markdown(instructions_text))

display(Markdown('## docs/AI_CLASSIFICATON_IS421_2026S.md'))
display(Markdown(assignment_text))

## 3) Goal #6 Config: Parse/Record Requirements

We capture the Goal #6 requirements into a machine-readable JSON file for reproducibility.

In [ ]:
goal6_requirements = {
    'goal': 'Goal #6 — Major-Class Classification (Scopus + Patents)',
    'data_sources': [
        'raw_data/MANI_KW_PAPERS_scopus.csv',
        'raw_data/MANI_KW_PATENTS_A_weds1969to2009.csv',
        'raw_data/MANI_KW_PATENTS_B_weds2010tonow.csv',
    ],
    'classification_basis': 'Abstract only (Title/Keywords/CPC/metadata must not influence classification)',
    'taxonomy_source': 'classification/MAJOR_CLASSES.md',
    'required_outputs': [
        'Serial Number',
        'Year',
        'Title',
        'Primary Major Class',
        'Primary Major Label',
        'Primary Major Score',
        'Secondary Major Class',
        'Secondary Major Label',
        'Secondary Major Score',
        'Tertiary Major Class',
        'Tertiary Major Label',
        'Tertiary Major Score',
    ],
    'integrity_constraint': 'All original columns must remain in final dataset',
    'sorting': ['Year', 'Primary Major Class', 'Secondary Major Class', 'Tertiary Major Class'],
    'modeling_constraints': {
        'no_paid_apis': True,
        'open_source_huggingface': True,
        'colab_free_tier': True,
    },
}

req_path = ROOT / 'goal6_requirements.json'
req_path.write_text(json.dumps(goal6_requirements, indent=2), encoding='utf-8')
print(f'Wrote requirements JSON: {req_path}')
req_path

## 4) Dataset Ingest: Load Scopus + Merge Patents (and clean)

We load the Scopus CSV and **append** the two patent CSVs (A + B) into one merged patents dataset. Cleaning is intentionally minimal and based on the goal constraints:

- Validate required columns exist (Scopus: `Year`, `Title`, `Abstract`; Patents: `Publication Year`, `Title`, `Abstract`).
- Compute `Year_num` by numeric coercion; rows with missing/non-numeric year are dropped and labeled `invalid_year`.
- Compute `Abstract_clean` by filling nulls and stripping whitespace; rows with empty abstracts are dropped and labeled `empty_abstract`.
- For auditability, write a drop-report CSV per source under `outputs/goal6/` with a per-row `DropReason` (including `invalid_year+empty_abstract` when both apply).
- Add `Serial Number` after dropping rows (1..N) for stable referencing.

In [ ]:
def _compute_drop_reason(*, invalid_year: pd.Series, empty_abstract: pd.Series) -> pd.Series:
    # Produces one explicit reason string per row (empty string means keep).
    return pd.Series(
        np.select(
            [invalid_year & empty_abstract, invalid_year, empty_abstract],
            ['invalid_year+empty_abstract', 'invalid_year', 'empty_abstract'],
            default='',
        ),
        index=invalid_year.index,
    )


def _write_drop_report(*, dropped_df: pd.DataFrame, out_path: Path, source_label: str) -> None:
    if len(dropped_df) == 0:
        print(f"{source_label} drop report: no dropped rows")
        return

    out_path.parent.mkdir(parents=True, exist_ok=True)
    dropped_df.to_csv(out_path, index=False)
    print(f"Wrote {source_label} drop report: {out_path} ({len(dropped_df)} rows)")
    display(
        dropped_df[['DropReason', 'Year', 'Title']]
        .head(10)
        .reset_index(drop=True)
    )


def load_scopus_dataframe() -> pd.DataFrame:
    scopus_path = Path(CONFIG['scopus_csv'])
    if not scopus_path.exists():
        raise FileNotFoundError(f'Missing Scopus CSV: {scopus_path}')

    df_raw = pd.read_csv(scopus_path, dtype=str)
    required_input_cols = ['Year', 'Title', 'Abstract']
    missing_input = [c for c in required_input_cols if c not in df_raw.columns]
    if missing_input:
        raise ValueError(f'Missing required columns in Scopus CSV: {missing_input}')

    df = df_raw.copy()
    df['Year_num'] = pd.to_numeric(df['Year'], errors='coerce')
    df['Abstract_clean'] = df['Abstract'].fillna('').astype(str).str.strip()

    invalid_year = df['Year_num'].isna()
    empty_abstract = df['Abstract_clean'].str.len() == 0
    drop_any = invalid_year | empty_abstract

    df['_drop_reason'] = _compute_drop_reason(invalid_year=invalid_year, empty_abstract=empty_abstract)

    dropped_df = df.loc[drop_any].copy()
    dropped_df.insert(0, 'DropReason', dropped_df.pop('_drop_reason'))

    before = len(df)
    df = df.loc[~drop_any].copy()

    print('Scopus dropped rows:', before - len(df))
    print('Scopus drop reasons:', dropped_df['DropReason'].value_counts().to_dict())
    print('Scopus clean rows:', len(df))

    drop_path = Path(CONFIG['out_dir']) / 'goal6_scopus_dropped_rows.csv'
    _write_drop_report(dropped_df=dropped_df, out_path=drop_path, source_label='Scopus')

    df.insert(0, 'Serial Number', np.arange(1, len(df) + 1))
    df['Year_num'] = df['Year_num'].astype(int)
    df.drop(columns=['_drop_reason'], inplace=True, errors='ignore')
    return df


def load_patents_dataframe() -> pd.DataFrame:
    pat_a = Path(CONFIG['patents_csv_a'])
    pat_b = Path(CONFIG['patents_csv_b'])
    for p in [pat_a, pat_b]:
        if not p.exists():
            raise FileNotFoundError(f'Missing patent CSV: {p}')

    df_a = pd.read_csv(pat_a, dtype=str)
    df_b = pd.read_csv(pat_b, dtype=str)
    df_raw = pd.concat([df_a, df_b], ignore_index=True)
    print('Patents A shape:', df_a.shape)
    print('Patents B shape:', df_b.shape)
    print('Patents merged shape:', df_raw.shape)

    required_input_cols = ['Publication Year', 'Title', 'Abstract']
    missing_input = [c for c in required_input_cols if c not in df_raw.columns]
    if missing_input:
        raise ValueError(f'Missing required columns in patent CSVs: {missing_input}')

    df = df_raw.copy()
    df['Year'] = df['Publication Year']
    df['Year_num'] = pd.to_numeric(df['Year'], errors='coerce')
    df['Abstract_clean'] = df['Abstract'].fillna('').astype(str).str.strip()

    invalid_year = df['Year_num'].isna()
    empty_abstract = df['Abstract_clean'].str.len() == 0
    drop_any = invalid_year | empty_abstract

    df['_drop_reason'] = _compute_drop_reason(invalid_year=invalid_year, empty_abstract=empty_abstract)

    dropped_df = df.loc[drop_any].copy()
    dropped_df.insert(0, 'DropReason', dropped_df.pop('_drop_reason'))

    before = len(df)
    df = df.loc[~drop_any].copy()

    print('Patents dropped rows:', before - len(df))
    print('Patents drop reasons:', dropped_df['DropReason'].value_counts().to_dict())
    print('Patents clean rows:', len(df))

    drop_path = Path(CONFIG['out_dir']) / 'goal6_patents_dropped_rows.csv'
    _write_drop_report(dropped_df=dropped_df, out_path=drop_path, source_label='Patents')

    df.insert(0, 'Serial Number', np.arange(1, len(df) + 1))
    df['Year_num'] = df['Year_num'].astype(int)
    df.drop(columns=['_drop_reason'], inplace=True, errors='ignore')
    return df


df_scopus = load_scopus_dataframe()
df_patents = load_patents_dataframe()

print('Scopus head:')
display(df_scopus[['Serial Number', 'Year', 'Title']].head())
print('Patents head:')
display(df_patents[['Serial Number', 'Year', 'Title']].head())

## 5) Parse the 5 Major Classes from `classification/MAJOR_CLASSES.md`

We parse the definitions and build clean candidate labels for zero-shot classification.

In [ ]:
major_md_path = Path(CONFIG['major_classes_md'])
if not major_md_path.exists():
    raise FileNotFoundError(f'Missing major classes markdown: {major_md_path}')

major_text = major_md_path.read_text(encoding='utf-8', errors='ignore')

MAJOR_ORDER = ['MATERIAL', 'COMPUTATION', 'EXPERIMENTATION', 'APPLICATION', 'REVIEW / BOOK']
MAJOR_CODE = {name: i + 1 for i, name in enumerate(MAJOR_ORDER)}

def _norm_major(name: str) -> str:
    # Normalize headings to improve robustness to minor edits.
    n = name.strip().upper()
    n = n.replace('/', ' / ')
    n = re.sub(r'\s+', ' ', n).strip()
    return n

def parse_major_classes(markdown_text: str) -> list[dict]:
    majors: dict[str, dict] = {}
    current = None

    for raw in markdown_text.splitlines():
        line = raw.rstrip()
        stripped = line.strip()

        # Section headers: ## MATERIAL, etc.
        if stripped.startswith('## '):
            name_raw = stripped.replace('## ', '', 1).strip()
            name = _norm_major(name_raw)
            current = name
            majors[current] = {
                'major': name,
                'definition': '',
            }
            continue

        if current is None:
            continue

        if stripped.startswith('**Definition:**'):
            definition = stripped.replace('**Definition:**', '', 1).strip()
            majors[current]['definition'] = definition

    # Normalize + order
    out = []
    for name in MAJOR_ORDER:
        if name not in majors:
            raise ValueError(f'Major class not found in MAJOR_CLASSES.md: {name}')
        d = majors[name]
        out.append({
            'code': MAJOR_CODE[name],
            'major': name,
            'definition': re.sub(r'\s+', ' ', d.get('definition', '')).strip(),
        })

    # Candidate label for model: keep it short but informative
    for c in out:
        maj = c['major']
        if maj == 'REVIEW / BOOK':
            label_name = 'REVIEW/BOOK'
        else:
            label_name = maj

        if c['definition']:
            c['label'] = f"{label_name}: {c['definition']}"
        else:
            c['label'] = f"{label_name}"

    return out

major_classes = parse_major_classes(major_text)
pd.DataFrame(major_classes)

## 6) Classifier: Abstract-only Zero-shot Classification (5 labels)

We apply an open-source MNLI zero-shot classifier over the 5 major classes.

In [ ]:
MODEL_NAME = CONFIG['model_name']
device = 0 if torch.cuda.is_available() else -1
print('CUDA available:', torch.cuda.is_available(), '| device:', device)

zero_shot = pipeline(
    task='zero-shot-classification',
    model=MODEL_NAME,
    device=device,
)

candidate_labels = [c['label'] for c in major_classes]
label_to_meta = {c['label']: c for c in major_classes}

print('Initialized zero-shot model:', MODEL_NAME)
print('Candidate labels:', len(candidate_labels))
candidate_labels

## 7) Run Classification for Scopus + Patents

This section runs the classifier over **abstract text only** and produces top-3 major-class predictions.

In [ ]:
def _top3_from_result_major(result: dict) -> dict:
    labels = result['labels'][:3]
    scores = result['scores'][:3]
    metas = [label_to_meta[l] for l in labels]

    primary = metas[0]
    secondary = metas[1] if len(metas) > 1 else metas[0]
    tertiary = metas[2] if len(metas) > 2 else metas[-1]

    primary_score = float(scores[0]) if len(scores) > 0 else None
    secondary_score = float(scores[1]) if len(scores) > 1 else primary_score
    tertiary_score = float(scores[2]) if len(scores) > 2 else (float(scores[-1]) if len(scores) > 0 else None)

    def _label_name(m):
        return 'REVIEW/BOOK' if m['major'] == 'REVIEW / BOOK' else m['major']

    return {
        'Primary Major Class': int(primary['code']),
        'Primary Major Label': _label_name(primary),
        'Primary Major Score': primary_score,
        'Secondary Major Class': int(secondary['code']),
        'Secondary Major Label': _label_name(secondary),
        'Secondary Major Score': secondary_score,
        'Tertiary Major Class': int(tertiary['code']),
        'Tertiary Major Label': _label_name(tertiary),
        'Tertiary Major Score': tertiary_score,
    }


def classify_abstracts_zero_shot(texts: list[str], batch_size: int):
    # Efficiency note: Transformers recommends using `datasets` to avoid slow per-item calls.
    # This runs the pipeline over a Dataset-backed iterable while still batching on the backend.
    from datasets import Dataset
    from transformers.pipelines.pt_utils import KeyDataset

    ds = Dataset.from_dict({'text': texts})
    outputs = []
    iterator = zero_shot(
        KeyDataset(ds, 'text'),
        candidate_labels=candidate_labels,
        multi_label=False,
        hypothesis_template=CONFIG['hypothesis_template'],
        truncation=True,
        batch_size=int(batch_size),
    )
    for out in tqdm(iterator, total=len(ds)):
        outputs.append(out)
    return outputs


def run_major_classification(df_in: pd.DataFrame, dataset_name: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    batch_size = int(CONFIG['batch_size'])
    abstracts = df_in['Abstract_clean'].tolist()
    print(f'Starting major-class classification for {dataset_name}: {len(abstracts)} abstracts (batch_size={batch_size})')

    results = classify_abstracts_zero_shot(abstracts, batch_size=batch_size)
    classified_rows = [_top3_from_result_major(r) for r in results]
    classified_df = pd.DataFrame(classified_rows)

    # Diagnostics
    primary_counts = classified_df['Primary Major Class'].value_counts(dropna=False).sort_index()
    primary_counts_df = primary_counts.rename('count').reset_index().rename(columns={'index': 'Primary Major Class'})

    # Ambiguity score: margin between top1 and top2 (primary vs secondary)
    classified_df['margin_12'] = classified_df['Primary Major Score'] - classified_df['Secondary Major Score']
    ambiguous = classified_df.nsmallest(25, 'margin_12')[[
        'Primary Major Class', 'Secondary Major Class',
        'Primary Major Score', 'Secondary Major Score',
        'margin_12',
    ]]

    return classified_df, primary_counts_df, ambiguous


classified_scopus, scopus_counts, scopus_ambiguous = run_major_classification(df_scopus, 'scopus')
classified_patents, patents_counts, patents_ambiguous = run_major_classification(df_patents, 'patents')

display(scopus_counts)
display(patents_counts)

## 8) Save Artifacts (CSVs + metrics + config)

We export per-dataset CSVs while preserving all original columns, and write useful diagnostics under `outputs/goal6/`.

In [ ]:
def export_major_dataset(df_in: pd.DataFrame, classified_df: pd.DataFrame, dataset_name: str, out_csv: str):
    export_cols = [
        'Primary Major Class', 'Primary Major Label', 'Primary Major Score',
        'Secondary Major Class', 'Secondary Major Label', 'Secondary Major Score',
        'Tertiary Major Class', 'Tertiary Major Label', 'Tertiary Major Score',
    ]
    classified_export = classified_df[export_cols].copy()

    df_out = pd.concat([df_in.reset_index(drop=True), classified_export.reset_index(drop=True)], axis=1)

    # Sort by: Year, Primary, Secondary, Tertiary
    for col in ['Primary Major Class', 'Secondary Major Class', 'Tertiary Major Class']:
        df_out[col] = pd.to_numeric(df_out[col], errors='coerce')

    df_out = df_out.sort_values(
        by=['Year_num', 'Primary Major Class', 'Secondary Major Class', 'Tertiary Major Class'],
        ascending=True,
    ).reset_index(drop=True)

    df_out = df_out.drop(columns=['Year_num', 'Abstract_clean'], errors='ignore')

    out_path = Path(out_csv)
    df_out.to_csv(out_path, index=False)
    print(f'Wrote {dataset_name} CSV:', out_path)

    required_cols = goal6_requirements['required_outputs']
    missing = [c for c in required_cols if c not in df_out.columns]
    print(f'Missing required columns ({dataset_name}):', missing)
    return df_out


# Export main deliverables
df_out_scopus = export_major_dataset(df_scopus, classified_scopus, 'scopus', CONFIG['out_csv_scopus'])
df_out_patents = export_major_dataset(df_patents, classified_patents, 'patents', CONFIG['out_csv_patents'])

# Save metrics
metrics_dir = Path(CONFIG['out_dir'])
scopus_counts.to_csv(metrics_dir / 'goal6_scopus_metrics_primary_major_class_counts.csv', index=False)
scopus_ambiguous.to_csv(metrics_dir / 'goal6_scopus_metrics_ambiguous_top25.csv', index=False)
patents_counts.to_csv(metrics_dir / 'goal6_patents_metrics_primary_major_class_counts.csv', index=False)
patents_ambiguous.to_csv(metrics_dir / 'goal6_patents_metrics_ambiguous_top25.csv', index=False)

# Save edge cases joined back to original rows
def edge_cases_table(df_in: pd.DataFrame, ambiguous_df: pd.DataFrame) -> pd.DataFrame:
    edge_idx = ambiguous_df.index.tolist()
    return pd.concat(
        [df_in.loc[edge_idx, ['Serial Number', 'Year', 'Title', 'Abstract']].reset_index(drop=True),
         ambiguous_df.reset_index(drop=True)],
        axis=1,
    )

edge_scopus = edge_cases_table(df_scopus, scopus_ambiguous)
edge_patents = edge_cases_table(df_patents, patents_ambiguous)
edge_scopus.to_csv(metrics_dir / 'goal6_scopus_edge_cases.csv', index=False)
edge_patents.to_csv(metrics_dir / 'goal6_patents_edge_cases.csv', index=False)

# Save config snapshot
config_path = metrics_dir / 'goal6_run_config.json'
config_path.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print('Also wrote:', config_path)
print('Done.')